In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import FunctionTransformer, StandardScaler, OneHotEncoder, PowerTransformer
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, root_mean_squared_error

# ==========================================
# 1. CREATE SKEWED SYNTHETIC DATA
# ==========================================
np.random.seed(42)
n_samples = 1000

# Generating heavily log-normal (skewed) numerical features
num_1 = np.random.lognormal(mean=1.5, sigma=0.7, size=n_samples)
num_2 = np.random.lognormal(mean=0.5, sigma=1.2, size=n_samples)
num_3 = np.random.normal(loc=10, scale=2, size=n_samples) # Not skewed

# Categorical feature
cat_1 = np.random.choice(['Low', 'Medium', 'High'], size=n_samples)

# Target variable 'amount' driven by these features
amount = (num_1 * 2) + (num_2 * 0.5) + (num_3 * 10) + np.random.normal(0, 5, n_samples)

df = pd.DataFrame({
    'num_1': num_1,
    'num_2': num_2,
    'num_3': num_3,
    'cat_1': cat_1,
    'amount': amount
})

# Separate features and target
X = df.drop(columns=['amount'])
y = df['amount']

num_cols = ['num_1', 'num_2', 'num_3']
cat_cols = ['cat_1']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# ==========================================
# 2. DEFINE THE TRANSFORM FUNCTION
# ==========================================
def fit_skewness(X):
    X = X.copy()
    for col in X.columns:
        if abs(X[col].skew()) > 0.5:
            pt = PowerTransformer(method="yeo-johnson")
            X[[col]] = pt.fit_transform(X[[col]])
    return X

# ==========================================
# 3. BUILD PIPELINE WITH SKEWNESS FIX (Model A)
# ==========================================
num_trans_with_skew = Pipeline([
    ('imp', SimpleImputer(strategy='mean')),
    ('skew', FunctionTransformer(fit_skewness, feature_names_out='one-to-one')),
    ('scr', StandardScaler())
])

cat_trans = Pipeline([
    ('imp', SimpleImputer(strategy='most_frequent')),
    ('ohe', OneHotEncoder(sparse_output=False, handle_unknown='ignore'))
])

preprocessor_with_skew = ColumnTransformer([
    ('num_pre', num_trans_with_skew, num_cols),
    ('cat_trans', cat_trans, cat_cols)
])
preprocessor_with_skew.set_output(transform="pandas")

model_with_skew = Pipeline([
    ('preprocessor', preprocessor_with_skew),
    ('regressor', RandomForestRegressor(random_state=42))
])

# Train and evaluate Model A
model_with_skew.fit(X_train, y_train)
preds_a = model_with_skew.predict(X_test)
r2_a = r2_score(y_test, preds_a)
rmse_a = root_mean_squared_error(y_test, preds_a)

# ==========================================
# 4. BUILD PIPELINE WITHOUT SKEWNESS FIX (Model B)
# ==========================================
num_trans_no_skew = Pipeline([
    ('imp', SimpleImputer(strategy='mean')),
    ('scr', StandardScaler()) # Skipped FunctionTransformer
])

preprocessor_no_skew = ColumnTransformer([
    ('num_pre', num_trans_no_skew, num_cols),
    ('cat_trans', cat_trans, cat_cols)
])
preprocessor_no_skew.set_output(transform="pandas")

model_no_skew = Pipeline([
    ('preprocessor', preprocessor_no_skew),
    ('regressor', RandomForestRegressor(random_state=42))
])

# Train and evaluate Model B
model_no_skew.fit(X_train, y_train)
preds_b = model_no_skew.predict(X_test)
r2_b = r2_score(y_test, preds_b)
rmse_b = root_mean_squared_error(y_test, preds_b)

# ==========================================
# 5. PRINT COMPARISON RESULTS
# ==========================================
print("=== PERFORMANCE COMPARISON ===")
print(f"Model WITH Skewness Fix -> R² Score: {r2_a:.4f} | RMSE: {rmse_a:.4f}")
print(f"Model WITHOUT Skewness Fix -> R² Score: {r2_b:.4f} | RMSE: {rmse_b:.4f}")


=== PERFORMANCE COMPARISON ===
Model WITH Skewness Fix -> R² Score: 0.8799 | RMSE: 8.2201
Model WITHOUT Skewness Fix -> R² Score: 0.8794 | RMSE: 8.2348
